In [ ]:
import numpy as np
import cvxpy as cp
import pandas as pd
from pathlib import Path

In [ ]:
cp.MAX_NUM_SUBEXPRESSIONS = 10000

- Change data into daily profiles => you want to get a distribution of battery sizes depending on the profile of days
    *modify daily profile function from daily profile clustering + incorporate aggregation function.

In [ ]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data/complete_dataframe2_30min.csv"
df = pd.read_csv(DATA_DIR, 
                     delimiter=',', 
                     decimal='.', 
                     parse_dates=['ts'],  
                     index_col='ts')

# Aggregate df across PV modules and wind turbines
df_agg = df.copy()
df_agg["PV_total"] = df_agg[["B117_m", "B319_m", "B330_1_m","B330_2_m","B330_3_m", "B716_m", "B715_m"]].sum(axis=1, min_count=1)
df_agg["Wind_total"] = df_agg[["Aircon_WT Power_m", "Gaia_WT Power_m"]].sum(axis=1, min_count= 1)
df_agg.dropna(subset=["PV_total", "Wind_total"], how='any')

#should I make the power data into daily profiles? 

n = df_agg["PV_total"].size 
print(n)
#current n size assumes we have info for a full year; n should span a single day with timesteps depending on the time resolution of the optimization (choice of smaller horizon should be explained);
#reduces complexity

Start optimization with real data to have comparison in terms of accuracy
    - assume you have next-day's weather and power values
Upon having comparison, you can incorporate clustering results
optimization code should be wrapped in a function
    input: 24-hour of real pv power and real wind
    output: schedule for battery operation

Load data may exist in Gaggle or we can simulate it from DTU's load data
    doesn't overlap directly timewise => don't take average so you have a spikier profile

In [ ]:
## Defining a constant 2 kW load
Pload_np = np.zeros((n))+2
Pload = cp.Constant(Pload_np)
dT = 0.25 #h

## Target min self-sufficiency
minimum_self_sufficiency = 0.8

## Battery parameters; power and energy capacities are different values :) -- efficiency refers to power, always
charge_eff = 0.95
discharge_eff = 0.95
P_capacity = cp.Variable()
E_capacity = cp.Variable()

## Defining generation values from Syslab Data -- separate into solar and wind generation
Pgen_PV = cp.Constant(df_agg['PV_total'])
Pgen_W = cp.Constant(df_agg['Wind_total'])
#print(df_agg['PV_total'].describe(), df_agg['Wind_total'].describe())

## Defining charge and discharge power values
Pcharge = cp.Variable(n,nonneg=True) # charge is positive
Pdischarge = cp.Variable(n,nonneg=True) # discharge is positive
z = cp.Variable(n, boolean = True) #boolean to prevent charging and discharging from happening simultaneously.

## Defining power imported and exported to the grid -- becomes 0 with 100% self-sufficiency
Pgrid_buy = cp.Variable(n, nonneg=True)
#Pgrid_sell = cp.Variable(n, nonneg=True) We're not selling anumore

## Defining variable SOC
SOC = cp.Variable(n)

## Defining equation for self-sufficiency 
self_sufficiency = (Pgen_PV + Pgen_W - Pcharge)/Pload 
average_self_sufficiency = sum(self_sufficiency)/n


In [ ]:
#To avoid simultaneous charge and discharge
constraints = [Pcharge[n-1] <= Pcharge * z[n-1],
               Pdischarge[n-1] <= Pdischarge * (1 - z[n-1])]
constraints = [Pcharge >= 0,
               Pdischarge >= 0,
               Pcharge <= P_capacity,
               Pdischarge <= P_capacity]
constraints += [SOC >= E_capacity * 0.1, SOC <= E_capacity] # SOC ranges between 10 to 100% across a day
constraints += [Pgen_PV + Pgen_W + Pdischarge - Pcharge + Pgrid_buy  == Pload, # power flow equation 
               SOC[0] == 0.5 * E_capacity + (Pcharge[0]*charge_eff - Pdischarge[0]/ discharge_eff)*dT, ## Starting with 50% charge
               SOC[1:] == SOC[:-1] + (Pcharge[1:] * charge_eff - Pdischarge[1:] / discharge_eff)*dT, # battery state definition
               SOC[n-1] >= 0.5 * E_capacity] # minimum 50% charge
               #average_self_sufficiency >= minimum_self_sufficiency # average self sufficiency over the year
               
               

In [ ]:
problem2 = cp.Problem(cp.Minimize(E_capacity), constraints) #minimize
problem2.solve(verbose=True)

Small size of battery could be due to:
- enough generation to not require battery
- lax constraint on self-sufficiency

Make sure that when you're charging your'e not discharding
    - maybe introduce boolean variable to prevent simultaneous operation

target function could be to minimize SOC * battery E capacity * 1.0 or just the E_capacity
you need to make the 